Team coursework: Random Forest classification on an anonymised Kaggle-style tabular dataset (class-imbalanced binary target), part of the same assignment as `svm_imbalance_and_tuning.ipynb` in this folder. The input CSV and rendered outputs are not included, and results are not independently reproduced in this portfolio copy.


In [ ]:
###EasyENSEMBLE WITH GRIDSEARCH

## LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, fbeta_score, matthews_corrcoef, precision_recall_curve, average_precision_score


## DATA PREP

# Import our data and save it as datafile
datafile = pd.read_csv("data/datafile_small.csv")

# Check the structure
datafile.info()

# Removing ID, Var_45
datafile = datafile.drop(['ID_code', 'var_45'], axis=1)


## PARTINIONING THE DATA AND THEN REPLACING MISSING VALUES WITH MEAN

randomseed = 30
X = datafile.drop("target", axis=1)
y = datafile["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=randomseed, stratify=y)

# Impute missing values with mean
imputer = SimpleImputer(strategy="mean")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Find proportion of target in the original dataset
print("Original Dataset target Proportions:")
print(datafile['target'].value_counts(normalize=True))

# Find proportion of target in the training set
print("\nTraining Set target Proportions:")
print(y_train.value_counts(normalize=True))

# Find proportion of target in the test set
print("\nTest Set target Proportions:")
print(y_test.value_counts(normalize=True))


## APPLYING EasyENSEMBLE UNDER-SAMPLING TECHNIQUE TO BALANCE THE TRAINING SET AND BUILDING RANDOM FOREST MODEL

# Build Random Forest model
model_RF = RandomForestClassifier(random_state=randomseed)

eec = EasyEnsembleClassifier(estimator=model_RF, random_state=randomseed)  
eec.fit(X_train, y_train)


## MODEL EVALUATION

# Evaluate accuracy on training set
train_pred = eec.predict(X_train)
print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred))

# Using model_RF predict the class of the test data
prediction_RF = eec.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF))

# Compute the confusion matrix
print("Testing Confusion Matrix - Random Forest:")
print(confusion_matrix(y_test, prediction_RF))

# Print the Classification Report
print("\nTesting Classification Report - Random Forest:")
print(classification_report(y_test, prediction_RF))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF, beta=2)
print("F2 Score - Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF)
print("MCC Score - Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec.predict_proba(X_test)[:,1])
print("PR-AUC Score (Average Precision) - Random Forest:", pr_auc)


## HYPERPARAMETER TUNING WITH GRID SEARCH

#Define parameter grid
param_grid_rf = {
    'n_estimators': [100,200],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_leaf': [1, 3, 5, 7, 9]
}

# Create GridSearchCV object
grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=randomseed),
    param_grid_rf,
    cv=3,
    scoring='average_precision',
    verbose=2 
)

# Fit the grid search
grid_search_rf.fit(X_train, y_train)

# Print the best parameters
print("Best Hyperparameters for Random Forest:")
print(f"\nBest parameters:")
print(f"n_estimators: {grid_search_rf.best_params_['n_estimators']}")
print(f"max_depth: {grid_search_rf.best_params_['max_depth']}")
print(f"min_samples_leaf: {grid_search_rf.best_params_['min_samples_leaf']}")

## BUILDING TUNED RF MODEL WITH EasyENSEMBLE BALANCING

# Build Random Forest model with best parameters
model_RF_tuned = RandomForestClassifier(random_state=randomseed, n_estimators=200, max_depth=30, min_samples_leaf=7)

eec_tuned = EasyEnsembleClassifier(estimator=model_RF_tuned, random_state=randomseed)  
eec_tuned.fit(X_train, y_train)

## MODEL EVALUATION TUNED

# Evaluate accuracy on training set
train_pred_tuned = eec_tuned.predict(X_train)
print("Training Accuracy:", accuracy_score(y_train, train_pred_tuned))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred_tuned))

# Using model_RF predict the class of the test data
prediction_RF_tuned = eec_tuned.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF_tuned))

# Compute the confusion matrix
print("Testing Confusion Matrix - Tuned Random Forest:")
print(confusion_matrix(y_test, prediction_RF_tuned))

# Print the Classification Report
print("\nTesting Classification Report - Tuned Random Forest:")
print(classification_report(y_test, prediction_RF_tuned))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec_tuned.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Tuned Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF_tuned, beta=2)
print("F2 Score - Tuned Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF_tuned)
print("MCC Score - Tuned Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec_tuned.predict_proba(X_test)[:,1])
print("PR-AUC Score - Tuned Random Forest:", pr_auc)



In [ ]:
###SMOTE

## LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, fbeta_score, matthews_corrcoef, precision_recall_curve
from imblearn.over_sampling import SMOTE

## DATA PREP

# Import our data and save it as datafile
datafile = pd.read_csv("data/datafile_small.csv")

# Check the structure
datafile.info()

# Removing ID, Var_45
datafile = datafile.drop(['ID_code', 'var_45'], axis=1)


## PARTINIONING THE DATA AND THEN REPLACING MISSING VALUES WITH MEAN

randomseed = 30
X = datafile.drop("target", axis=1)
y = datafile["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=randomseed, stratify=y)

# Impute missing values with mean
imputer = SimpleImputer(strategy="mean")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Find proportion of target in the original dataset
print("Original Dataset target Proportions:")
print(datafile['target'].value_counts(normalize=True))

# Find proportion of target in the training set
print("\nTraining Set target Proportions:")
print(y_train.value_counts(normalize=True))

# Find proportion of target in the test set
print("\nTest Set target Proportions:")
print(y_test.value_counts(normalize=True))


## APPLYING SMOTE TO BALANCE THE TRAINING SET

smote = SMOTE(random_state=randomseed)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# Show new class proportions
print("\nResampled Training Set target Proportions after SMOTE:")
print(y_resampled.value_counts(normalize=True))

# Final training data after SMOTE
X_train_final = X_resampled
y_train_final = y_resampled

## MODELLING WITH RANDOM FOREST

# Build Random Forest model
model_RF = RandomForestClassifier(random_state=randomseed)
model_RF.fit(X_train_final, y_train_final)

# Print model_RF
print("Random Forest Model:")
print(f"Number of trees: {model_RF.n_estimators}")
print(f"Number of features: {model_RF.n_features_in_}")
print(f"Training accuracy: {model_RF.score(X_train_final, y_train_final):.4f}")

# Using model_RF predict the class of the test data
prediction_RF = model_RF.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF))

# Compute the confusion matrix
print("Confusion Matrix - Random Forest:")
print(confusion_matrix(y_test, prediction_RF))

# Print the Classification Report
print("\nClassification Report - Random Forest:")
print(classification_report(y_test, prediction_RF))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, model_RF.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF, beta=2)
print("F2 Score - Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF)
print("MCC Score - Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, model_RF.predict_proba(X_test)[:,1])
print("PR-AUC Score - Random Forest:", pr_auc)

## HYPERPARAMETER TUNING USING GRID SEARCH

# Define parameter grid
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 20, 30, None],
   'min_samples_leaf': [1, 3, 5, 7, 10]
}

# Create GridSearchCV object
grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=randomseed),
    param_grid_rf,
    cv=3,
    scoring='average_precision',
    verbose=2 
)

# Fit the grid search
grid_search_rf.fit(X_train_final, y_train_final)

# Print the best parameters
print("Best Hyperparameters for Random Forest:")
print(f"\nBest parameters:")
print(f"n_estimators: {grid_search_rf.best_params_['n_estimators']}")
print(f"max_depth: {grid_search_rf.best_params_['max_depth']}")
print(f"min_samples_leaf: {grid_search_rf.best_params_['min_samples_leaf']}")

# BUILDING TUNED RF MODEL
model_RF_tuned = RandomForestClassifier(random_state=randomseed,n_estimators=200, max_depth=None, min_samples_leaf=1)
model_RF_tuned.fit(X_train_final, y_train_final)

# Print model_RF
print("Tuned Random Forest Model:")
print(f"Number of trees: {model_RF_tuned.n_estimators}")
print(f"Number of features: {model_RF_tuned.n_features_in_}")
print(f"Tuned Training accuracy: {model_RF_tuned.score(X_train_final, y_train_final):.4f}")

# Using model_RF predict the class of the test data
prediction_RF_tuned = model_RF_tuned.predict(X_test)
print("\nTuned Test Accuracy:", accuracy_score(y_test, prediction_RF_tuned))

# Compute the confusion matrix
print("Confusion Matrix - Tuned Random Forest:")
print(confusion_matrix(y_test, prediction_RF_tuned))

# Print the Classification Report
print("\nClassification Report - Tuned Random Forest:")
print(classification_report(y_test, prediction_RF_tuned))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, model_RF_tuned.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Tuned Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF_tuned, beta=2)
print("F2 Score - Tuned Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF_tuned)
print("MCC Score - Tuned Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, model_RF_tuned.predict_proba(X_test)[:,1])
print("PR-AUC Score - Tuned Random Forest:", pr_auc)



In [ ]:
###CLASS WEIGHT

## LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, fbeta_score, matthews_corrcoef, precision_recall_curve, average_precision_score
from imblearn.over_sampling import SMOTE

## DATA PREP

# Import our data and save it as datafile
datafile = pd.read_csv("data/datafile_small.csv")

# Check the structure
datafile.info()

# Removing ID, Var_45
datafile = datafile.drop(['ID_code', 'var_45'], axis=1)


## PARTINIONING THE DATA AND THEN REPLACING MISSING VALUES WITH MEAN

randomseed = 30
X = datafile.drop("target", axis=1)
y = datafile["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=randomseed, stratify=y)

# Impute missing values with mean
imputer = SimpleImputer(strategy="mean")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Find proportion of target in the original dataset
print("Original Dataset target Proportions:")
print(datafile['target'].value_counts(normalize=True))

# Find proportion of target in the training set
print("\nTraining Set target Proportions:")
print(y_train.value_counts(normalize=True))

# Find proportion of target in the test set
print("\nTest Set target Proportions:")
print(y_test.value_counts(normalize=True))


## MODELLING WITH RANDOM FOREST

# Build Random Forest model
model_RF = RandomForestClassifier(random_state=randomseed,class_weight='balanced')
model_RF.fit(X_train, y_train)

# Print model_RF
print("Random Forest Model:")
print(f"Number of trees: {model_RF.n_estimators}")
print(f"Number of features: {model_RF.n_features_in_}")
print(f"Training accuracy: {model_RF.score(X_train, y_train):.4f}")

## BUILDING RANDOM FOREST MODEL

# Using model_RF predict the class of the test data
prediction_RF = model_RF.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF))

# Compute the confusion matrix
print("Confusion Matrix - Random Forest:")
print(confusion_matrix(y_test, prediction_RF))

# Print the Classification Report
print("\nClassification Report - Random Forest:")
print(classification_report(y_test, prediction_RF))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, model_RF.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF, beta=2)
print("F2 Score - Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF)
print("MCC Score - Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, model_RF.predict_proba(X_test)[:,1])
print("PR-AUC Score - Random Forest:", pr_auc)



In [ ]:
### NO BALANCING

## LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, fbeta_score, matthews_corrcoef, precision_recall_curve, average_precision_score
from imblearn.over_sampling import SMOTE

## DATA PREP

# Import our data and save it as datafile
datafile = pd.read_csv(r"data/datafile_small.csv")

# Check the structure
datafile.info()

# Removing ID, Var_45
datafile = pd.read_csv("data/datafile_small.csv")


## PARTINIONING THE DATA AND THEN REPLACING MISSING VALUES WITH MEAN

randomseed = 30
X = datafile.drop("target", axis=1)
y = datafile["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=randomseed, stratify=y)

# Impute missing values with mean
imputer = SimpleImputer(strategy="mean")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Find proportion of target in the original dataset
print("Original Dataset target Proportions:")
print(datafile['target'].value_counts(normalize=True))

# Find proportion of target in the training set
print("\nTraining Set target Proportions:")
print(y_train.value_counts(normalize=True))

# Find proportion of target in the test set
print("\nTest Set target Proportions:")
print(y_test.value_counts(normalize=True))


## MODELLING WITH RANDOM FOREST

# Build Random Forest model
model_RF = RandomForestClassifier(random_state=randomseed)
model_RF.fit(X_train, y_train)

# Print model_RF
print("Random Forest Model:")
print(f"Number of trees: {model_RF.n_estimators}")
print(f"Number of features: {model_RF.n_features_in_}")
print(f"Training accuracy: {model_RF.score(X_train, y_train):.4f}")

# Using model_RF predict the class of the test data
prediction_RF = model_RF.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF))

# Compute the confusion matrix
print("Confusion Matrix - Random Forest:")
print(confusion_matrix(y_test, prediction_RF))

# Print the Classification Report
print("\nClassification Report - Random Forest:")
print(classification_report(y_test, prediction_RF))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, model_RF.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF, beta=2)
print("F2 Score - Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF)
print("MCC Score - Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, model_RF.predict_proba(X_test)[:,1])
print("PR-AUC Score - Random Forest:", pr_auc)


In [ ]:
###EasyENSEMBLE WITH OPTUNA

## LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, fbeta_score, matthews_corrcoef, precision_recall_curve, average_precision_score
import optuna 

## DATA PREP

# Import our data and save it as datafile
datafile = pd.read_csv("data/datafile_small.csv")

# Check the structure
datafile.info()

# Removing ID, Var_45
datafile = datafile.drop(['ID_code', 'var_45'], axis=1)


## PARTINIONING THE DATA AND THEN REPLACING MISSING VALUES WITH MEAN

randomseed = 30
X = datafile.drop("target", axis=1)
y = datafile["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=randomseed, stratify=y)

# Impute missing values with mean
imputer = SimpleImputer(strategy="mean")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Find proportion of target in the original dataset
print("Original Dataset target Proportions:")
print(datafile['target'].value_counts(normalize=True))

# Find proportion of target in the training set
print("\nTraining Set target Proportions:")
print(y_train.value_counts(normalize=True))

# Find proportion of target in the test set
print("\nTest Set target Proportions:")
print(y_test.value_counts(normalize=True))


## APPLYING EasyENSEMBLE UNDER-SAMPLING TECHNIQUE TO BALANCE THE TRAINING SET AND BUILDING RANDOM FOREST MODEL

# Build Random Forest model
model_RF = RandomForestClassifier(random_state=randomseed)

eec = EasyEnsembleClassifier(estimator=model_RF, random_state=randomseed)  
eec.fit(X_train, y_train)


## MODEL EVALUATION

# Evaluate accuracy on training set
train_pred = eec.predict(X_train)
print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred))

# Using model_RF predict the class of the test data
prediction_RF = eec.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF))

# Compute the confusion matrix
print("Testing Confusion Matrix - Random Forest:")
print(confusion_matrix(y_test, prediction_RF))

# Print the Classification Report
print("\nTesting Classification Report - Random Forest:")
print(classification_report(y_test, prediction_RF))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF, beta=2)
print("F2 Score - Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF)
print("MCC Score - Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec.predict_proba(X_test)[:,1])
print("PR-AUC Score - Random Forest:", pr_auc)

## HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    max_depth = trial.suggest_int('max_depth', 10, 30)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)

    model = RandomForestClassifier(
       n_estimators=n_estimators,
       max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
         random_state=randomseed
    )

    eec = EasyEnsembleClassifier(estimator=model, random_state=randomseed)
    eec.fit(X_train, y_train)
    y_pred = eec.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, y_pred)
    return pr_auc       

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print("Best Hyperparameters for Random Forest with Optuna:")
print(study.best_params)


## BUILDING TUNED RF MODEL WITH EasyENSEMBLE BALANCING
model_RF_tuned = RandomForestClassifier(random_state=randomseed, n_estimators=171, max_depth=30, min_samples_leaf=2)

eec_tuned = EasyEnsembleClassifier(estimator=model_RF_tuned, random_state=randomseed)  
eec_tuned.fit(X_train, y_train)

## MODEL EVALUATION TUNED

# Evaluate accuracy on training set
train_pred_tuned = eec_tuned.predict(X_train)
print("Training Accuracy:", accuracy_score(y_train, train_pred_tuned))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred_tuned))

# Using model_RF predict the class of the test data
prediction_RF_tuned = eec_tuned.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF_tuned))

# Compute the confusion matrix
print("Testing Confusion Matrix - Tuned Random Forest:")
print(confusion_matrix(y_test, prediction_RF_tuned))

# Print the Classification Report
print("\nTesting Classification Report - Tuned Random Forest:")
print(classification_report(y_test, prediction_RF_tuned))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec_tuned.predict_proba(X_test)[:,1])
print("AUC-ROC Score - Tuned Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF_tuned, beta=2)
print("F2 Score - Tuned Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF_tuned)
print("MCC Score - Tuned Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec_tuned.predict_proba(X_test)[:,1])
print("PR-AUC Score - Tuned Random Forest:", pr_auc)



In [ ]:
###EasyENSEMBLE WITH GRIDSEARCH AND FEATURE SELECTION

## LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, fbeta_score, matthews_corrcoef, precision_recall_curve, average_precision_score
from sklearn.feature_selection import mutual_info_classif


## DATA PREP

# Import our data and save it as datafile
datafile = pd.read_csv("data/datafile_small.csv")

# Check the structure
datafile.info()

# Removing ID, Var_45
datafile = datafile.drop(['ID_code', 'var_45'], axis=1)


## PARTINIONING THE DATA AND THEN REPLACING MISSING VALUES WITH MEAN

randomseed = 30
X = datafile.drop("target", axis=1)
y = datafile["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=randomseed, stratify=y)

# Impute missing values with mean
imputer = SimpleImputer(strategy="mean")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Find proportion of target in the original dataset
print("Original Dataset target Proportions:")
print(datafile['target'].value_counts(normalize=True))

# Find proportion of target in the training set
print("\nTraining Set target Proportions:")
print(y_train.value_counts(normalize=True))

# Find proportion of target in the test set
print("\nTest Set target Proportions:")
print(y_test.value_counts(normalize=True))


## FEATURE SELECTION USING RANDOM FOREST IMPORTANCE

# Calculate information gain for all features
IG_scores = mutual_info_classif(X_train, y_train, random_state=randomseed)

# Create a DataFrame with the results
weights = pd.DataFrame({
    'attr_importance': IG_scores
}, index=X_train.columns)

# Print weights
print("Information Gain (Mutual Information) Scores:")

# Sort the weights in decreasing order of information gain values
weights = weights.sort_values('attr_importance', ascending=False)
print(weights)

# Filter features where the information gain is not zero
features = weights[weights['attr_importance']  > 0].index.tolist()
print(f"Number of features with positive information gain: {len(features)}")
print("\nFeatures with positive information gain:")
print(features)


## APPLYING EasyENSEMBLE UNDER-SAMPLING TECHNIQUE TO BALANCE THE TRAINING SET AND BUILDING RANDOM FOREST MODEL

# Build Random Forest model
model_RF = RandomForestClassifier(random_state=randomseed)

eec = EasyEnsembleClassifier(estimator=model_RF, random_state=randomseed)  
eec.fit(X_train[features], y_train)

## MODEL EVALUATION

# Evaluate accuracy on training set
train_pred = eec.predict(X_train[features])
print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred))

# Using model_RF predict the class of the test data
prediction_RF = eec.predict(X_test[features])
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF))

# Compute the confusion matrix
print("Testing Confusion Matrix - Random Forest:")
print(confusion_matrix(y_test, prediction_RF))

# Print the Classification Report
print("\nTesting Classification Report - Random Forest:")
print(classification_report(y_test, prediction_RF))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec.predict_proba(X_test[features])[:,1])
print("AUC-ROC Score - Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF, beta=2)
print("F2 Score - Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF)
print("MCC Score - Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec.predict_proba(X_test[features])[:,1])
print("PR-AUC Score - Random Forest:", pr_auc)
    


## HYPERPARAMETER TUNING WITH GRID SEARCH

#Define parameter grid
param_grid_rf = {
    'n_estimators': [100,200],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_leaf': [1, 3, 5, 7, 9]
}

# Create GridSearchCV object
grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=randomseed),
    param_grid_rf,
    cv=3,
    scoring='average_precision',  # PR-AUC
    verbose=2 
)

# Fit the grid search
grid_search_rf.fit(X_train[features], y_train)

# Print the best parameters
print("Best Hyperparameters for Random Forest:")
print(f"\nBest parameters:")
print(f"n_estimators: {grid_search_rf.best_params_['n_estimators']}")
print(f"max_depth: {grid_search_rf.best_params_['max_depth']}")
print(f"min_samples_leaf: {grid_search_rf.best_params_['min_samples_leaf']}")

## BUILDING TUNED RF MODEL WITH EasyENSEMBLE BALANCING
model_RF_tuned = RandomForestClassifier(random_state=randomseed, n_estimators=200, max_depth=None, min_samples_leaf=5)

eec_tuned = EasyEnsembleClassifier(estimator=model_RF_tuned, random_state=randomseed)  
eec_tuned.fit(X_train[features], y_train)

## MODEL EVALUATION TUNED

# Evaluate accuracy on training set
train_pred_tuned = eec_tuned.predict(X_train[features])
print("Training Accuracy:", accuracy_score(y_train, train_pred_tuned))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred_tuned))

# Using model_RF predict the class of the test data
prediction_RF_tuned = eec_tuned.predict(X_test[features])
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF_tuned))

# Compute the confusion matrix
print("Testing Confusion Matrix - Tuned Random Forest:")
print(confusion_matrix(y_test, prediction_RF_tuned))

# Print the Classification Report
print("\nTesting Classification Report - Tuned Random Forest:")
print(classification_report(y_test, prediction_RF_tuned))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec_tuned.predict_proba(X_test[features])[:,1])
print("AUC-ROC Score - Tuned Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF_tuned, beta=2)
print("F2 Score - Tuned Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF_tuned)
print("MCC Score - Tuned Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec_tuned.predict_proba(X_test[features])[:,1])
print("PR-AUC Score - Tuned Random Forest:", pr_auc)


In [ ]:
###EasyENSEMBLE WITH OPTUNA AND FEATURE SELECTION

## LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split, GridSearchCV
from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, roc_auc_score, accuracy_score, fbeta_score, matthews_corrcoef, precision_recall_curve, average_precision_score
import optuna 
from sklearn.feature_selection import mutual_info_classif


## DATA PREP

# Import our data and save it as datafile
datafile = pd.read_csv("data/datafile_small.csv")

# Check the structure
datafile.info()

# Removing ID, Var_45
datafile = datafile.drop(['ID_code', 'var_45'], axis=1)


## PARTINIONING THE DATA AND THEN REPLACING MISSING VALUES WITH MEAN

randomseed = 30
X = datafile.drop("target", axis=1)
y = datafile["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=randomseed, stratify=y)

# Impute missing values with mean
imputer = SimpleImputer(strategy="mean")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# Find proportion of target in the original dataset
print("Original Dataset target Proportions:")
print(datafile['target'].value_counts(normalize=True))

# Find proportion of target in the training set
print("\nTraining Set target Proportions:")
print(y_train.value_counts(normalize=True))

# Find proportion of target in the test set
print("\nTest Set target Proportions:")
print(y_test.value_counts(normalize=True))

## FEATURE SELECTION USING RANDOM FOREST IMPORTANCE

# Calculate information gain for all features
IG_scores = mutual_info_classif(X_train, y_train, random_state=randomseed)

# Create a DataFrame with the results
weights = pd.DataFrame({
    'attr_importance': IG_scores
}, index=X_train.columns)

# Print weights
print("Information Gain (Mutual Information) Scores:")

# Sort the weights in decreasing order of information gain values
weights = weights.sort_values('attr_importance', ascending=False)
print(weights)

# Filter features where the information gain is not zero
features = weights[weights['attr_importance']  > 0].index.tolist()
print(f"Number of features with positive information gain: {len(features)}")
print("\nFeatures with positive information gain:")
print(features)


## APPLYING EasyENSEMBLE UNDER-SAMPLING TECHNIQUE TO BALANCE THE TRAINING SET AND BUILDING RANDOM FOREST MODEL

# Build Random Forest model
model_RF = RandomForestClassifier(random_state=randomseed)

eec = EasyEnsembleClassifier(estimator=model_RF, random_state=randomseed)  
eec.fit(X_train[features], y_train)


## MODEL EVALUATION

# Evaluate accuracy on training set
train_pred = eec.predict(X_train[features])
print("Training Accuracy:", accuracy_score(y_train, train_pred))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred))

# Using model_RF predict the class of the test data
prediction_RF = eec.predict(X_test[features])
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF))

# Compute the confusion matrix
print("Testing Confusion Matrix - Random Forest:")
print(confusion_matrix(y_test, prediction_RF))

# Print the Classification Report
print("\nTesting Classification Report - Random Forest:")
print(classification_report(y_test, prediction_RF))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec.predict_proba(X_test[features])[:,1])
print("AUC-ROC Score - Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF, beta=2)
print("F2 Score - Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF)
print("MCC Score - Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec.predict_proba(X_test[features])[:,1])
print("PR-AUC Score - Random Forest:", pr_auc)


## HYPERPARAMETER TUNING WITH OPTUNA
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    max_depth = trial.suggest_int('max_depth', 10, 30)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)

    model = RandomForestClassifier(
       n_estimators=n_estimators,
       max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
         random_state=randomseed
    )

    eec = EasyEnsembleClassifier(estimator=model, random_state=randomseed)
    eec.fit(X_train[features], y_train)
    y_pred = eec.predict_proba(X_test[features])[:,1]
    pr_auc = average_precision_score(y_test, y_pred)
    return pr_auc       

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print("Best Hyperparameters for Random Forest with Optuna:")
print(study.best_params)


## BUILDING TUNED RF MODEL WITH EasyENSEMBLE BALANCING
model_RF_tuned = RandomForestClassifier(random_state=randomseed, n_estimators=150, max_depth=30, min_samples_leaf=2)

eec_tuned = EasyEnsembleClassifier(estimator=model_RF_tuned, random_state=randomseed)  
eec_tuned.fit(X_train[features], y_train)

## MODEL EVALUATION TUNED

# Evaluate accuracy on training set
train_pred_tuned = eec_tuned.predict(X_train[features])
print("Training Accuracy:", accuracy_score(y_train, train_pred_tuned))
print("\nTraining Classification Report:")
print(classification_report(y_train, train_pred_tuned))

# Using model_RF predict the class of the test data
prediction_RF_tuned = eec_tuned.predict(X_test[features])
print("\nTest Accuracy:", accuracy_score(y_test, prediction_RF_tuned))

# Compute the confusion matrix
print("Testing Confusion Matrix - Tuned Random Forest:")
print(confusion_matrix(y_test, prediction_RF_tuned))

# Print the Classification Report
print("\nTesting Classification Report - Tuned Random Forest:")
print(classification_report(y_test, prediction_RF_tuned))

#Print the AUC-ROC Score
roc_auc = roc_auc_score(y_test, eec_tuned.predict_proba(X_test[features])[:,1])
print("AUC-ROC Score - Tuned Random Forest:", roc_auc)

#Print the F2 Score
f2_score = fbeta_score(y_test, prediction_RF_tuned, beta=2)
print("F2 Score - Tuned Random Forest:", f2_score)

#Print the MCC Score
mcc_score = matthews_corrcoef(y_test, prediction_RF_tuned)
print("MCC Score - Tuned Random Forest:", mcc_score)  

#Print the PR-AUC Score
pr_auc = average_precision_score(y_test, eec_tuned.predict_proba(X_test[features])[:,1])
print("PR-AUC Score - Tuned Random Forest:", pr_auc)

